# Tarea Práctica - Módulo 6 PySpark

**Sistemas de Computación Distribuida - FAE Usach**
**Profesor: Ivan Espinoza**

---

## Datos del alumno

| Campo | Valor |
|-------|-------|
| Nombre completo | _completa aquí_ |
| Apellido | _completa aquí_ |
| RUT | _completa aquí_ |
| Email | _completa aquí_ |
| Pareja (si aplica) | _completa aquí_ |
| Fecha entrega | _completa aquí_ |

---

## Instrucciones generales

1. Esta plantilla tiene el **setup ya armado** (descarga del dataset y SparkSession). No necesitas modificarla.
2. Cada ejercicio tiene celdas con `# TODO:` donde debes escribir tu código.
3. Antes de entregar:
   - Ejecuta **todo el notebook desde cero** (Runtime > Restart and run all).
   - Verifica que **todas las celdas se ejecuten sin errores**.
   - Verifica que **al final no quede ningun stream activo**.
4. Guarda el notebook con el nombre `Tarea_M6_<apellido>_<nombre>.ipynb` y envíalo por correo a ivan.espinoza.m@gmail.com.


---
## Setup (no modificar)

### Instalación de PySpark


In [ ]:
!pip install pyspark==3.5.1 --quiet
print("PySpark instalado")

### Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Tarea_M6_FAE_USACH")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista. Versión:", spark.version)

### Descargar el dataset MovieLens

In [ ]:
# Descargamos y descomprimimos el dataset (~1 MB)
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /tmp/ml.zip
!unzip -q -o /tmp/ml.zip -d /tmp/
print("Archivos disponibles:")
!ls /tmp/ml-latest-small/

**Rutas de los archivos** (úsalas en los ejercicios):

```python
PATH_MOVIES  = "/tmp/ml-latest-small/movies.csv"
PATH_RATINGS = "/tmp/ml-latest-small/ratings.csv"
PATH_TAGS    = "/tmp/ml-latest-small/tags.csv"
PATH_LINKS   = "/tmp/ml-latest-small/links.csv"
```


In [ ]:
PATH_MOVIES  = "/tmp/ml-latest-small/movies.csv"
PATH_RATINGS = "/tmp/ml-latest-small/ratings.csv"
PATH_TAGS    = "/tmp/ml-latest-small/tags.csv"
PATH_LINKS   = "/tmp/ml-latest-small/links.csv"
print("Variables definidas")

---
# PARTE A - Fundamentos (Clase 2)

## Ejercicio 1 - Carga y exploración (10 pts)

Carga los archivos `movies.csv` y `ratings.csv` con **schema explícito**. Verifica:
- `printSchema` de ambos.
- `count()` de ambos.
- `show(5)` de ambos.

**Tipos sugeridos:**
- movies: `movieId` (int), `title` (string), `genres` (string).
- ratings: `userId` (int), `movieId` (int), `rating` (double), `timestamp` (long).

**Esperado:** movies ~9.700 filas, ratings ~100.000 filas.


In [ ]:
# TODO: define los schemas
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, LongType

schema_movies = ...   # TODO

schema_ratings = ...  # TODO

# TODO: lee los CSV con los schemas
df_movies = ...     # TODO

df_ratings = ...    # TODO

# TODO: verifica con printSchema, count y show



## Ejercicio 2 - Transformaciones básicas (15 pts)

### Parte a) y b) - Fechas en ratings

- Convierte `timestamp` (epoch seconds) a un `TimestampType`. Llama a la columna **`fecha`**.
- Agrega una columna **`anio_rating`** con el año del rating.

### Parte c) y d) - Limpieza de títulos en movies

- Crea **`titulo_limpio`**: el título sin el año ni los paréntesis. Ej: 'Toy Story'.
- Crea **`anio_pelicula`**: el año de la película como **integer**. Ej: 1995.
- Muestra el top 10 películas con el **titulo_limpio más largo** (usa `length`).


In [ ]:
from pyspark.sql.functions import col, from_unixtime, year, regexp_extract, regexp_replace, length

# TODO: parte a y b - agregar columna fecha y anio_rating a df_ratings
df_ratings = ...  # TODO

# TODO: parte c - agregar titulo_limpio y anio_pelicula a df_movies
df_movies = ...   # TODO

# TODO: parte d - top 10 con titulo_limpio más largo



## Ejercicio 3 - Agregaciones simples (15 pts)

Calcula por película:
- cantidad de ratings (count)
- rating promedio (avg)
- rating mínimo y máximo

Filtra solo películas con **al menos 50 ratings** (así los promedios son confiables).

Muestra el **top 10 mejor evaluadas** (con su título limpio del Ejercicio 2).


In [ ]:
from pyspark.sql.functions import avg, count, min as f_min, max as f_max, round as f_round

# TODO: agregar por movieId
df_stats = ...  # TODO

# TODO: filtrar cantidad >= 50

# TODO: hacer join con movies para tener el titulo_limpio

# TODO: mostrar top 10



---
# PARTE B - Joins y transformaciones avanzadas (Clase 3)

## Ejercicio 4 - Análisis de géneros (15 pts)

La columna `genres` viene como string con géneros separados por `|`. Una película puede tener varios géneros.

- **a)** Crea `df_peli_genero` con UNA fila por (película, género). Tip: `split` + `explode`.
- **b)** Top 10 géneros por cantidad de películas.
- **c)** Une `df_peli_genero` con ratings, calcula:
  - ratings totales por género
  - rating promedio por género
- **d)** Top 5 géneros con mejor promedio, **considerando solo géneros con > 1.000 ratings totales**.


In [ ]:
from pyspark.sql.functions import split, explode

# TODO: parte a - df_peli_genero (una fila por película y género)
df_peli_genero = ...  # TODO

# TODO: parte b - top 10 géneros por cantidad de películas

# TODO: parte c - join con ratings y agregaciones por género
df_stats_genero = ...  # TODO

# TODO: parte d - top 5 géneros con mejor promedio (> 1000 ratings)



## Ejercicio 5 - Window functions y escritura particionada (20 pts)

### Parte a) - Top 3 películas por género

Para cada género, ranking de las **3 películas con mejor rating promedio** (entre las que tengan al menos 30 ratings).

Esperado: una tabla con `género`, `titulo_limpio`, `promedio`, `posicion` (1, 2 o 3). 3 filas por género.

### Parte b) - Evolución temporal

Rating promedio agrupado por (`anio_pelicula`, `género`). Filtra películas desde 1980 en adelante.

### Parte c) - Escritura particionada

Escribe el resultado de (b) particionado por `género` en Parquet en `/tmp/ratings_por_genero/`. Verifica con `ls`.


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# TODO: parte a - ranking por género (top 3)
# necesitas: stats por (movieId, género) -> filter >= 30 ratings -> join con movies para título
# -> Window.partitionBy('genero').orderBy(col('promedio').desc())
# -> row_number() y filter <= 3



In [ ]:
# TODO: parte b - evolución temporal (anio_pelicula, género) desde 1980
df_evolucion = ...  # TODO



In [ ]:
# TODO: parte c - escribir particionado por género
# df_evolucion.write.mode('overwrite').partitionBy('genero').parquet('/tmp/ratings_por_genero')

# Verificar
# !ls /tmp/ratings_por_genero/ | head -10



---
# PARTE C - Streaming (Clase 4)

## Ejercicio 6 - Simulacion de stream de ratings (20 pts)

### Estrategia

1. Particionamos `ratings` por `anio_rating` en archivos JSONL.
2. Un thread los copia uno por uno a una carpeta destino con pausas (simulando que llegan en vivo).
3. Un `readStream` los procesa.

### Pasos

**a)** Particiona ratings en archivos JSONL por año. Cada archivo: `/tmp/ratings_por_anio/ratings_AAAA.jsonl`.

**b)** Define un productor que copia los archivos uno a uno hacia `/tmp/ratings_stream/` con pausas de 3 segundos.

**c)** Lanza un `readStream` sobre `/tmp/ratings_stream/` con schema explícito.

**d)** Calcula en vivo el **rating promedio y cantidad por película**. Output mode `update`. Memory sink.

**e)** Después de 30 segundos, consulta y muestra el top 10 películas por cantidad de ratings acumulados. Detener el stream.


In [ ]:
import os, shutil, json, time, threading
import pandas as pd

# Limpieza
for d in ["/tmp/ratings_por_anio", "/tmp/ratings_stream"]:
    if os.path.exists(d):
        shutil.rmtree(d)
os.makedirs("/tmp/ratings_por_anio", exist_ok=True)
os.makedirs("/tmp/ratings_stream", exist_ok=True)
print("Carpetas limpias")

In [ ]:
# TODO: parte a) particionar ratings por anio_rating
# Para cada año distinto, escribir un JSONL en /tmp/ratings_por_anio/ratings_AAAA.jsonl
# Tip: puedes hacer .toPandas() y usar pandas.to_json(..., orient='records', lines=True)
# (es OK porque el dataset es chico, ~100k filas)

# anios_unicos = sorted([r[0] for r in df_ratings.select('anio_rating').distinct().collect()])
# for anio in anios_unicos:
#     ...



In [ ]:
# TODO: parte b) productor que copia archivos uno a uno

# def productor(...):
#     for archivo in sorted(os.listdir('/tmp/ratings_por_anio')):
#         shutil.copy(...)
#         time.sleep(3)



In [ ]:
# TODO: parte c) definir el readStream
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType, StringType, TimestampType

# schema_stream = StructType([...])

# df_stream = (
#     spark.readStream
#     .schema(schema_stream)
#     .option("maxFilesPerTrigger", 1)
#     .json("/tmp/ratings_stream")
# )



In [ ]:
# TODO: parte d) calcular agregado y lanzar a memoria

# df_agg = df_stream.groupBy('movieId').agg(...)

# query = (
#     df_agg.writeStream
#     .format("memory")
#     .queryName("ratings_vivo")
#     .outputMode("update")
#     .start()
# )



In [ ]:
# TODO: parte e) lanzar el productor en thread, esperar 30s, consultar, detener

# t = threading.Thread(target=productor, daemon=True)
# t.start()
# time.sleep(30)
# spark.sql("SELECT * FROM ratings_vivo ORDER BY cantidad DESC LIMIT 10").show()
# query.stop()



---
# Bonus opcional (+10 pts)

Elige **una** de las dos opciones (no acumulables):

## Opcion A - Tag analysis

Lee `tags.csv` y encuentra:
- Top 10 tags más frecuentes.
- Para esas 10 tags: rating promedio de las películas asociadas.

## Opcion B - Stream con ventana temporal

En el Ejercicio 6, en vez del agregado global, calcula **rating promedio por ventana tumbling de 5 segundos**. Muestra las últimas 5 ventanas.


In [ ]:
# (Opcional) Tu solución del bonus aquí



---
# Cierre - detener todo y cerrar SparkSession

Esta celda es obligatoria. Si la dejas fuera te resto puntos por dejar streams activos.


In [ ]:
# Detener todos los streams activos por las dudas
queries_activas = [q for q in spark.streams.active]
print(f"Queries activas a detener: {len(queries_activas)}")
for q in queries_activas:
    q.stop()
    print(f"  Detenida: {q.id}")

spark.stop()
print("SparkSession cerrada. Tarea terminada.")